# UrbanFleet Logistics – Decision Tree Classification Model
### Delivery Delay Risk Prediction
---

## Task 1: Understand the Business Problem

### What is the business about?
UrbanFleet Logistics is a local delivery company offering same-day and next-day delivery services across several city regions. Its customers include online retailers, small businesses, corporate offices, and marketplace sellers who depend on reliable delivery to maintain customer satisfaction.

### What problem is the business trying to solve?
UrbanFleet faces a key operational challenge: some orders are more likely to be delivered late, but managers do not always know which orders need extra attention **before** they leave the warehouse. Late deliveries reduce customer satisfaction and may lead to refunds, service credits, or lost future orders. The company also risks wasting resources by manually reviewing orders that are not actually high-risk, while genuinely risky orders may be missed when conditions are busy.

### What decision can machine learning help the business make?
Machine learning can help UrbanFleet decide **whether a delivery order should be flagged for proactive intervention before dispatch** — for example, assigning an experienced driver, adjusting the route, or contacting the customer early about a potential delay.

### What is the target variable in the dataset?
The target variable is **`Late_Delivery_Risk`** — a binary label indicating whether an order is predicted to have a high risk (`Yes`) or low risk (`No`) of late delivery.

### What are the input features?
The input features cover four groups:
- **Customer & order information:** `Customer_Type`, `Product_Category`, `Order_Value_USD`
- **Delivery geography:** `Delivery_Region`, `Distance_Km`
- **Operational conditions:** `Warehouse_Load_Level`, `Dispatch_Hour`, `Delivery_Window_Hours`, `Package_Weight_Kg`
- **External & historical conditions:** `Weather_Condition`, `Traffic_Level`, `Weekend_Order`, `Previous_Delays_For_Customer`, `Driver_Experience_Years`, `Priority_Shipping`

### Why is this prediction useful for the business?
By predicting high-risk orders before dispatch, UrbanFleet can allocate resources more efficiently, prevent customer dissatisfaction, reduce refund costs, and maintain its reliability reputation — all without reviewing every single order manually.

---
## Task 2: Prepare the Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

print("Libraries imported successfully.")

In [ ]:
# ── Load the dataset ──────────────────────────────────────────────────────────
df = pd.read_excel(
    "urbanfleet_delivery_delay_risk_dataset.xlsx",
    sheet_name="Delivery_Data"
)

print("Dataset loaded successfully.")
df.head()

In [ ]:
# ── Inspect the dataset ───────────────────────────────────────────────────────
print("Shape (rows, columns):", df.shape)
print("\nColumn data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# ── Check missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

In [ ]:
# ── Check duplicate rows ──────────────────────────────────────────────────────
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# ── Check target variable distribution ───────────────────────────────────────
print("Target variable distribution:")
print(df["Late_Delivery_Risk"].value_counts())
print()
print(df["Late_Delivery_Risk"].value_counts(normalize=True).round(3))

In [ ]:
# ── Clean the dataset ─────────────────────────────────────────────────────────
# Drop identifier and date columns (not useful as predictive features)
df_clean = df.drop(columns=["Order_ID", "Order_Date"])

# Fill missing numerical values with median
df_clean["Driver_Experience_Years"] = df_clean["Driver_Experience_Years"].fillna(
    df_clean["Driver_Experience_Years"].median()
)

# Fill missing categorical values with mode
df_clean["Weather_Condition"] = df_clean["Weather_Condition"].fillna(
    df_clean["Weather_Condition"].mode()[0]
)
df_clean["Traffic_Level"] = df_clean["Traffic_Level"].fillna(
    df_clean["Traffic_Level"].mode()[0]
)

print("Missing values after cleaning:")
print(df_clean.isnull().sum().sum(), "total missing values")

In [ ]:
# ── Separate features and target variable ────────────────────────────────────
# Encode target: Yes -> 1, No -> 0
y = (df_clean["Late_Delivery_Risk"] == "Yes").astype(int)

# Features (drop target column)
X_raw = df_clean.drop(columns=["Late_Delivery_Risk"])

print("Target variable (y) sample:")
print(y.value_counts())

In [ ]:
# ── Handle categorical variables with one-hot encoding ───────────────────────
categorical_cols = [
    "Customer_Type", "Delivery_Region", "Product_Category",
    "Warehouse_Load_Level", "Weather_Condition", "Traffic_Level",
    "Priority_Shipping", "Weekend_Order"
]

X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=False)

print("Feature matrix shape after encoding:", X.shape)
print("\nFeature columns:")
print(list(X.columns))

In [ ]:
# ── Split the data into training and testing sets (80 / 20) ──────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size : {X_train.shape[0]} rows")
print(f"Testing set size  : {X_test.shape[0]} rows")

---
## Task 3: Train a Decision Tree Classification Model

In [ ]:
# ── Train the Decision Tree ───────────────────────────────────────────────────
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

print("Decision Tree trained successfully.")

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_train = dt_model.predict(X_train)
y_pred_test  = dt_model.predict(X_test)

In [ ]:
# ── Training and Testing Accuracy ────────────────────────────────────────────
train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test,  y_pred_test)

print(f"Training Accuracy : {train_acc:.4f}  ({train_acc*100:.1f}%)")
print(f"Testing  Accuracy : {test_acc:.4f}  ({test_acc*100:.1f}%)")

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_test)
print("Confusion Matrix (Test Set):")
print(cm)
print()
print("  Rows = Actual class | Columns = Predicted class")
print("  [0,0] True Negatives  (TN):", cm[0,0])
print("  [0,1] False Positives (FP):", cm[0,1])
print("  [1,0] False Negatives (FN):", cm[1,0])
print("  [1,1] True Positives  (TP):", cm[1,1])

# Plot
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Risk", "High Risk"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix – Decision Tree (Test Set)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ── Precision, Recall, F1-Score ──────────────────────────────────────────────
precision = precision_score(y_test, y_pred_test)
recall    = recall_score(y_test,    y_pred_test)
f1        = f1_score(y_test,        y_pred_test)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print("Full Classification Report:")
print(classification_report(y_test, y_pred_test, target_names=["No Risk (0)", "High Risk (1)"]))

In [ ]:
# ── Summary metrics table ────────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    "Metric": ["Training Accuracy", "Testing Accuracy", "Precision", "Recall", "F1-Score"],
    "Value":  [train_acc, test_acc, precision, recall, f1]
})
metrics_df["Value"] = metrics_df["Value"].round(4)
print(metrics_df.to_string(index=False))

In [ ]:
# ── Feature Importance Chart ─────────────────────────────────────────────────
feature_importance = pd.Series(
    dt_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 15 Feature Importances:")
print(feature_importance.head(15).round(4))

fig, ax = plt.subplots(figsize=(9, 6))
feature_importance.head(15).sort_values().plot(
    kind="barh", ax=ax, color="steelblue", edgecolor="white"
)
ax.set_title("Top 15 Feature Importances – Decision Tree", fontsize=13, fontweight='bold')
ax.set_xlabel("Importance Score", fontsize=11)
ax.set_ylabel("Feature", fontsize=11)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

### Overfitting Discussion

The Decision Tree achieves a **training accuracy of 100%** but a **testing accuracy of only 61.1%**. This is a very large gap of approximately 39 percentage points, which is a clear and strong sign of **overfitting**.

An unconstrained Decision Tree will keep splitting nodes until every training sample is perfectly classified. This means the model has essentially memorised the training data — including its noise and outliers — rather than learning general patterns that apply to new, unseen orders.

As a result, the model performs poorly on the test set. The low precision (0.25) and recall (0.28) on the high-risk class confirm that the model struggles to correctly identify late-delivery risk in new data. To address overfitting in practice, hyperparameters such as `max_depth`, `min_samples_split`, or `min_samples_leaf` should be tuned, or pruning techniques should be applied.

---
## Task 4: Business Interpretation of Model Results

---

### How well did the Decision Tree model perform?

The overall testing accuracy was **61.1%**, which is only slightly better than randomly guessing the majority class (No Risk = 71.1% of all records). More importantly, the model's performance on the minority class — identifying **high-risk orders** — was weak: precision of **0.25** means that only 1 in 4 orders flagged as high-risk was truly risky, and recall of **0.28** means the model only detected about 1 in 4 genuinely late deliveries. This level of performance would provide limited operational value to UrbanFleet in its current state.

---

### Is there a large difference between training accuracy and testing accuracy?

Yes. Training accuracy was **100%** while testing accuracy was **61.1%** — a gap of approximately **39 percentage points**. This is one of the largest possible gaps and confirms severe overfitting.

---

### Does the model show signs of overfitting? Why or why not?

Yes, the model shows very strong signs of overfitting. The default `DecisionTreeClassifier` in scikit-learn grows a fully unpruned tree, which creates extremely specific rules that fit the training data perfectly (100% accuracy) but fail to generalise to new orders (61.1%). The tree has memorised the training set rather than discovered genuine business patterns.

To reduce overfitting, UrbanFleet's data team should constrain the tree using parameters such as `max_depth` (e.g., 5–10), `min_samples_split`, or `min_samples_leaf`. Alternatively, an ensemble method like Random Forest or Gradient Boosting would likely perform significantly better.

---

### Which evaluation metric is most important for this business problem?

**Recall** is the most important metric for UrbanFleet. The primary risk is **missing a high-risk order** — a false negative means a late delivery goes undetected, leading to customer dissatisfaction, a refund, and potentially a lost account. The cost of a missed intervention (unhappy customer, refund, reputational damage) is typically higher than the cost of a false alarm (minor extra staff effort on an order that would have arrived on time anyway).

That said, if UrbanFleet has limited operational capacity for interventions, precision becomes increasingly important too — too many false alarms will waste driver re-routing and dispatch resources. The **F1-score** would then be the appropriate balance metric.

---

### What do false positives mean in this business context?

A **false positive** occurs when the model predicts an order is at high risk of late delivery, but the order would actually arrive on time.

**Business impact:** UrbanFleet would spend extra resources on that order unnecessarily — for example, assigning a more senior driver, proactively contacting the customer, or rerouting the delivery. While this is a wasted effort, it is generally a low-cost mistake. The customer experience is not harmed (they may even appreciate the communication), and the only real cost is staff time and potential route inefficiency.

With 15 false positives in the test set (out of 20 predicted high-risk orders), the current model flags many orders incorrectly, which could burden operations if deployed as-is.

---

### What do false negatives mean in this business context?

A **false negative** occurs when the model predicts an order is safe, but the order is actually late.

**Business impact:** UrbanFleet misses a genuinely risky delivery, takes no action, and the customer receives a late package with no prior warning. This leads to customer complaints, requests for refunds or service credits, possible social media complaints, and — for business customers — potential loss of the account.

With 13 false negatives in the test set (out of 18 truly high-risk orders), the current model misses the majority of late deliveries — meaning it provides very little protective value for UrbanFleet.

---

### Which features were most important in the Decision Tree model?

The top features by importance score were:

| Rank | Feature | Importance |
|------|---------|------------|
| 1 | Package_Weight_Kg | 0.2149 |
| 2 | Dispatch_Hour | 0.1284 |
| 3 | Order_Value_USD | 0.1198 |
| 4 | Distance_Km | 0.1182 |
| 5 | Driver_Experience_Years | 0.0912 |
| 6 | Weather_Condition_Rain | 0.0756 |
| 7 | Previous_Delays_For_Customer | 0.0310 |

Note: Because the model is overfitted, these importances reflect patterns in the training data only and should be interpreted with caution until a properly tuned model is built.

---

### How can these important features help the business make better decisions?

- **Package_Weight_Kg (most important):** Heavier packages take longer to load, handle, and deliver. UrbanFleet should consider adjusted delivery time estimates for heavy orders and ensure that drivers assigned to these orders have appropriate vehicle capacity.

- **Dispatch_Hour:** Orders dispatched late in the day are more likely to be delayed, possibly because traffic peaks or warehouse capacity is strained in the afternoon. The business could set a cut-off time for same-day delivery commitments and prioritise early-dispatching for high-value or fragile orders.

- **Distance_Km:** Longer delivery distances are naturally riskier. UrbanFleet could introduce a tiered delivery promise — same-day only within a certain radius — and proactively communicate longer windows for outer-region deliveries.

- **Driver_Experience_Years:** Less experienced drivers appear linked to higher delay risk. Pairing newer drivers with simpler, shorter routes and assigning experienced drivers to complex or long-distance deliveries could reduce late deliveries.

- **Weather_Condition_Rain:** Rain is a significant external risk factor. Integrating real-time weather data into the dispatch system and dynamically adjusting delivery windows on rainy days would allow UrbanFleet to set realistic customer expectations.

- **Previous_Delays_For_Customer:** Customers with a history of receiving delayed orders may live in harder-to-reach areas or have complex delivery requirements. The business can proactively flag these accounts for special handling.

---

### What is one possible limitation or bias in the model or dataset?

A key limitation is **class imbalance**. The dataset contains 256 "No Risk" orders (71%) and only 104 "High Risk" orders (29%). Without any balancing technique (e.g., oversampling with SMOTE or class weighting), the Decision Tree is biased toward predicting the majority class. This explains the model's poor recall on high-risk orders — it has learnt that predicting "No Risk" is statistically safe, even when a delivery is genuinely at risk.

A second limitation is **data recency and scope**. The dataset is based on historical and simulated data. Sudden real-world events — road closures, extreme weather, driver shortages — are not captured and cannot be predicted by a static model trained on past records.

There is also a risk of **geographic and customer-type bias**: if certain regions or customer segments historically received worse service, the model may perpetuate those patterns by continuing to deprioritise them rather than flag them for improvement.

---

### Why should human judgment still be used when making business decisions based on model results?

The Decision Tree is a **decision-support tool**, not an automatic decision-maker. Human judgment remains essential for several reasons:

1. **Model limitations:** The current model is severely overfitted and has low recall. Acting purely on its predictions would leave most late deliveries undetected.

2. **Context the model cannot see:** Real-time events — a driver calling in sick, a sudden storm, a road closure — are not in the training data and cannot be predicted by any historical model.

3. **Fairness and ethics:** If the model learns to flag certain regions or customer types more often due to historical operational bias rather than genuine risk, blindly following the model could unfairly disadvantage some customers.

4. **Customer relationships:** Deciding how to communicate a potential delay — and when to offer compensation — requires empathy, judgment about customer importance, and brand awareness that no model can provide.

Managers should review high-risk predictions in context, combine them with current operational knowledge, and make decisions that are fair, practical, and customer-focused.